# Lab: Shor's Algorithm with Qiskit

In this lab we implement **Shor's factoring algorithm**, the quantum algorithm that factors integers exponentially faster than the best known classical algorithms. The lab follows the lecture slides on Shor's algorithm: the quantum core is **order finding**, so we build that first and then use it to factor.

Given a positive integer $N$ and an integer $a$ with $\gcd(a, N) = 1$, the **order of $a$ modulo $N$** is the smallest positive integer $r$ such that

$$a^r \equiv 1 \pmod N.$$

Order finding is believed to be hard for classical computers, but it can be solved in polynomial time on a quantum computer using **quantum phase estimation (QPE)**. Shor's algorithm reduces factoring to order finding:

1. Pick a base $a$ coprime to $N$.
2. Find the order $r$ of $a$ modulo $N$ with a quantum computer.
3. If $r$ is even and $a^{r/2} \not\equiv \pm 1 \pmod N$, then $\gcd(a^{r/2} - 1, N)$ is a nontrivial factor of $N$.
4. Otherwise, retry with a different base.

In this lab we will:

1. Implement classical order finding and check the examples from the lecture.
2. Build the modular-multiplication oracle $M_a$.
3. Construct and verify the eigenstates of $M_a$.
4. Build the order-finding circuit from controlled powers of $M_a$ and the inverse QFT.
5. Recover the order from a phase-estimation measurement using continued fractions.
6. Implement full quantum order finding.
7. Factor integers by combining order finding with the classical reduction above.

> **Scope:** all arithmetic is done with small moduli and exact statevector simulation. We represent $M_a$ with a `UnitaryGate`, which is convenient for learning the algorithm but is not how it would be compiled efficiently on a large quantum computer. The lecture's final caveats discuss that scaling step.

> **Convention:** Qiskit uses **little-endian** ordering. Qubit 0 is the least significant bit, and a computational basis state $|x\rangle$ has statevector index $x$.

In [1]:
from fractions import Fraction
from math import gcd

import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Operator, Statevector

print("Qiskit imported successfully!")

Qiskit imported successfully!


### Helper Functions & Guardrails

The helpers below are used throughout the lab and by the test cells:

- `validate_modulus(N)` checks that $N \geq 3$ is an integer.
- `validate_base(a, N)` checks that $1 \leq a < N$ and $\gcd(a, N) = 1$.
- `target_qubit_count(N)` returns $m = \lceil \log_2 N \rceil$, the number of qubits needed to hold $0, \dots, N-1$.
- `basis_state_circuit(n, x)` prepares the computational basis state $|x\rangle$.
- `qft(n)` and `inverse_qft(n)` build the QFT and its adjoint.
- `control_probabilities(circuit, n_count)` simulates a circuit and aggregates the exact output probabilities by the measurement outcome of the control register.
- `reduce_order(a, N, r)` removes unnecessary factors from a candidate order.
- `is_prime(n)` and `perfect_power(n)` are the classical preprocessing helpers used by the full factoring routine.

> **Qubit layout for phase estimation:** the control register is qubits $0, \dots, n-1$ and the target register is qubits $n, \dots, n+m-1$. Because Qiskit displays basis states with the highest-index qubit first, the control register is the **last** $n$ characters of a bitstring and the target register is the first $m$ characters.

In [2]:
def validate_modulus(N: int) -> None:
    """Guardrail: checks that N is an integer with N >= 3."""
    assert isinstance(N, int) and N >= 3, "N must be an integer with N >= 3."


def validate_base(a: int, N: int) -> None:
    """Guardrail: checks that 1 <= a < N and gcd(a, N) = 1."""
    validate_modulus(N)
    assert isinstance(a, int) and 1 <= a < N, "a must be an integer with 1 <= a < N."
    assert gcd(a, N) == 1, f"a={a} and N={N} must be coprime."


def validate_qubit_count(n: int) -> None:
    """Guardrail: checks that n is a positive integer."""
    assert isinstance(n, int) and n >= 1, "n must be a positive integer."


def target_qubit_count(N: int) -> int:
    """Returns the number of qubits needed to hold the values 0, ..., N - 1."""
    validate_modulus(N)
    return max(1, (N - 1).bit_length())


def basis_state_circuit(n: int, x: int) -> QuantumCircuit:
    """Prepares the n-qubit computational basis state |x>."""
    validate_qubit_count(n)
    assert isinstance(x, int) and 0 <= x < 2 ** n, (
        f"x must be an integer in [0, 2**n), got {x}."
    )

    circuit = QuantumCircuit(n)
    for qubit in range(n):
        if (x >> qubit) & 1:
            circuit.x(qubit)
    return circuit


def qft(n: int) -> QuantumCircuit:
    """Builds the n-qubit quantum Fourier transform circuit."""
    validate_qubit_count(n)

    circuit = QuantumCircuit(n)
    for i in range(n - 1, -1, -1):
        circuit.h(i)
        for j in range(i - 1, -1, -1):
            circuit.cp(2 * np.pi / 2 ** (i - j + 1), j, i)

    for i in range(n // 2):
        circuit.swap(i, n - 1 - i)
    return circuit


def inverse_qft(n: int) -> QuantumCircuit:
    """Builds the n-qubit inverse quantum Fourier transform circuit."""
    return qft(n).inverse()


def control_probabilities(circuit: QuantumCircuit, n_count: int) -> dict[int, float]:
    """
    Returns the exact probabilities of the control-register outcomes.

    The control register is qubits 0, ..., n_count-1, so in Qiskit's
    display order it is the last n_count characters of each bitstring.
    Outcomes with probability below 1e-12 are treated as numerical noise.
    """
    validate_qubit_count(n_count)
    assert circuit.num_qubits > n_count, (
        "The circuit must have at least one target qubit."
    )

    m = circuit.num_qubits - n_count
    probabilities = Statevector(circuit).probabilities_dict()

    aggregated: dict[int, float] = {}
    for bitstring, probability in probabilities.items():
        if probability < 1e-12:
            continue
        outcome = int(bitstring[m:], 2)
        aggregated[outcome] = aggregated.get(outcome, 0.0) + float(probability)
    return aggregated


def reduce_order(a: int, N: int, r: int) -> int:
    """Removes factors from r while a**(r/f) = 1 (mod N) still holds."""
    validate_base(a, N)
    assert isinstance(r, int) and r >= 1, "r must be a positive integer."
    assert pow(a, r, N) == 1, f"a**r must be 1 (mod N), but r={r}."

    divisor = 2
    while divisor * divisor <= r:
        while r % divisor == 0 and pow(a, r // divisor, N) == 1:
            r //= divisor
        divisor += 1
    return r


def is_prime(n: int) -> bool:
    """Trial-division primality test, sufficient for the small moduli here."""
    if n < 2:
        return False
    if n % 2 == 0:
        return n == 2

    divisor = 3
    while divisor * divisor <= n:
        if n % divisor == 0:
            return False
        divisor += 2
    return True


def perfect_power(n: int) -> int | None:
    """Returns a base b with b**e = n for some e >= 2, or None."""
    for exponent in range(2, n.bit_length() + 1):
        root = round(n ** (1 / exponent))
        for candidate in (root - 1, root, root + 1):
            if candidate >= 2 and candidate ** exponent == n:
                return candidate
    return None

## Task 1: Classical Order Finding

The classical problem statement from the lecture is:

- **Input:** positive integers $N$ and $a$ satisfying $\gcd(N, a) = 1$.
- **Output:** the smallest positive integer $r$ such that $a^r \equiv 1 \pmod N$.

The integer $r$ is called the **order** of $a$ modulo $N$.

For example:

- $a = 3$, $N = 5$: the powers are $3, 4, 2, 1$, so $r = 4$.
- $a = 2$, $N = 5$: the powers are $2, 4, 3, 1$, so $r = 4$.

Two trivial examples from the lecture are:

- $a = 1$, $N = p$ for any prime $p$, where $r = 1$.
- $a = p - 1$, $N = p$ for any prime $p$, where $r = 2$ because $(p-1)^2 \equiv 1 \pmod p$.

### Task 1.1: Implement Classical Order Finding

Implement `classical_order(a: int, N: int) -> int`.

- `a` and `N` satisfy the guardrail conditions above.
- Returns the smallest positive integer $r$ with `pow(a, r, N) == 1`.

> **Hint:** start with `value = 1` and repeatedly multiply by `a` modulo `N`. The first time the value returns to `1`, the number of multiplications is the order.

In [5]:
def classical_order(a: int, N: int) -> int:
    """
    Returns the multiplicative order of a modulo N.

    Args:
        a: The base, with 1 <= a < N and gcd(a, N) = 1.
        N: The modulus, with N >= 3.

    Returns:
        int: The smallest positive integer r such that a**r = 1 (mod N).
    """
    validate_base(a, N)

    # TODO: multiply by a modulo N until the value returns to 1
    value = 1
    r = 0
    while True:
        value = (value * a) % N
        r += 1
        if value == 1:
            return r

### Task 1.2: Test Classical Order Finding

The cell below checks the lecture examples, the two trivial families, and several larger cases. It also checks that $r$ is the **smallest** exponent, not merely an exponent that happens to work.

In [6]:
CASES = [
    (3, 5, 4),
    (2, 5, 4),
    (1, 7, 1),
    (6, 7, 2),
    (2, 7, 3),
    (3, 7, 6),
    (2, 9, 6),
    (4, 9, 3),
    (2, 15, 4),
    (4, 15, 2),
    (2, 21, 6),
    (4, 21, 3),
    (2, 35, 12),
]

for a, N, expected in CASES:
    observed = classical_order(a, N)
    assert observed == expected, (
        f"classical_order(a={a}, N={N}) returned {observed}, expected {expected}."
    )

for p in [3, 5, 7, 11, 13]:
    assert classical_order(1, p) == 1, f"Order of 1 modulo {p} should be 1."
    assert classical_order(p - 1, p) == 2, (
        f"Order of {p - 1} modulo {p} should be 2."
    )

print("Classical order finding passed all tests!")

Classical order finding passed all tests!


## Task 2: The Modular-Multiplication Oracle

The quantum problem statement replaces the classical "multiply and reduce" step with a reversible oracle. For a base $a$ coprime to $N$, define

$$M_a |x\rangle = |a x \bmod N\rangle.$$

The target register has $m = \lceil \log_2 N \rceil$ qubits, so it can hold every value from $0$ to $2^m - 1$. Since $N$ is not necessarily a power of two, the lecture defines the oracle on the extra basis states by

$$M_a |x\rangle =
\begin{cases}
|a x \bmod N\rangle, & 0 \le x < N, \\
|x\rangle, & N \le x < 2^m.
\end{cases}$$

This makes $M_a$ a permutation of the $2^m$ basis states, hence a unitary.

Applying the oracle repeatedly gives

$$(M_a)^i |x\rangle = |a^i x \bmod N\rangle.$$

In particular, starting from $|1\rangle$ produces the sequence

$$|1\rangle,\ |a\rangle,\ |a^2 \bmod N\rangle,\ \dots,\ |a^{r-1} \bmod N\rangle,\ |1\rangle.$$

### Task 2.1: Implement the Matrix and the Oracle

Implement both functions:

- `modular_multiplication_matrix(a, N)` returns the $2^m \times 2^m$ matrix of $M_a$, with $m = $ `target_qubit_count(N)`.
- `modular_multiplication_oracle(a, N)` returns an $m$-qubit `QuantumCircuit` implementing that matrix.

> **Hint:** build the matrix column by column. Column $x$ has a single `1` in row $(a x) \bmod N$ when $x < N$, and in row $x$ otherwise. Wrap the matrix in a `UnitaryGate` to build the circuit.

In [7]:
def modular_multiplication_matrix(a: int, N: int) -> np.ndarray:
    """
    Returns the matrix of the oracle M_a: |x> -> |a*x mod N>.

    Args:
        a: The base, with 1 <= a < N and gcd(a, N) = 1.
        N: The modulus, with N >= 3.

    Returns:
        np.ndarray: The 2**m by 2**m unitary matrix of M_a, where
        m = target_qubit_count(N).
    """
    validate_base(a, N)

    # TODO: build the permutation matrix of M_a
    m = target_qubit_count(N)
    dimension = 2 ** m
    matrix = np.zeros((dimension, dimension), dtype=complex)
    for x in range(dimension):
        target = (a * x) % N if x < N else x
        matrix[target, x] = 1
    return matrix


def modular_multiplication_oracle(a: int, N: int) -> QuantumCircuit:
    """
    Builds the m-qubit circuit implementing M_a.

    Args:
        a: The base, with 1 <= a < N and gcd(a, N) = 1.
        N: The modulus, with N >= 3.

    Returns:
        QuantumCircuit: An m-qubit circuit implementing M_a.
    """
    validate_base(a, N)

    # TODO: wrap modular_multiplication_matrix(a, N) in a UnitaryGate
    m = target_qubit_count(N)
    circuit = QuantumCircuit(m)
    circuit.append(UnitaryGate(modular_multiplication_matrix(a,N)), range(m))
    return circuit

### Task 2.2: Test the Oracle

The cell below checks that:

- the matrix is unitary and acts correctly on every basis state,
- the circuit implements exactly the same matrix,
- $M_a^r = I$, and
- repeated applications of the oracle send $|1\rangle$ to $|a^i \bmod N\rangle$.

In [8]:
ORACLE_CASES = [(2, 5), (3, 5), (2, 15), (4, 15), (2, 21)]

for a, N in ORACLE_CASES:
    m = target_qubit_count(N)
    dimension = 2 ** m
    matrix = modular_multiplication_matrix(a, N)

    assert matrix.shape == (dimension, dimension), (
        f"M_{a} mod {N} has shape {matrix.shape}, expected "
        f"({dimension}, {dimension})."
    )
    assert np.allclose(matrix.conj().T @ matrix, np.eye(dimension)), (
        f"M_{a} mod {N} is not unitary."
    )

    oracle = modular_multiplication_oracle(a, N)
    assert oracle.num_qubits == m, (
        f"Oracle for a={a}, N={N} acts on {oracle.num_qubits} qubits, expected {m}."
    )
    assert np.allclose(Operator(oracle).data, matrix), (
        f"The circuit for a={a}, N={N} does not implement the matrix."
    )

    for x in range(dimension):
        expected = (a * x) % N if x < N else x
        observed = int(np.argmax(np.abs(matrix[:, x])))
        assert observed == expected, (
            f"M_{a} mod {N} maps |{x}> to |{observed}>, expected |{expected}>."
        )

    r = classical_order(a, N)
    assert np.allclose(np.linalg.matrix_power(matrix, r), np.eye(dimension)), (
        f"M_{a} mod {N} does not return to the identity after r={r} steps."
    )

    for i in range(r + 1):
        power = np.linalg.matrix_power(matrix, i)
        expected_state = np.eye(dimension)[pow(a, i, N)]
        assert np.allclose(power[:, 1], expected_state), (
            f"(M_{a})^{i} |1> is not |{pow(a, i, N)}> for N={N}."
        )

        circuit = basis_state_circuit(m, 1)
        for _ in range(i):
            circuit.compose(oracle, inplace=True)
        observed = int(np.argmax(np.abs(Statevector(circuit).data)))
        assert observed == pow(a, i, N), (
            f"Applying the oracle {i} times to |1> gave |{observed}>, "
            f"expected |{pow(a, i, N)}>."
        )

print("Modular-multiplication oracle passed all tests!")

Modular-multiplication oracle passed all tests!


## Task 3: Eigenstates of $M_a$

Quantum phase estimation needs an eigenstate of the unitary it is estimating. The oracle $M_a$ has a beautiful set of eigenstates built from the powers of $a$.

Let $r$ be the order of $a$ modulo $N$, and define, for $s = 0, 1, \dots, r-1$,

$$|u_s\rangle = \frac{1}{\sqrt r} \sum_{k=0}^{r-1} e^{-2\pi i s k / r} |a^k \bmod N\rangle.$$

Because $M_a |a^k \bmod N\rangle = |a^{k+1} \bmod N\rangle$ (with the index taken modulo $r$), each $|u_s\rangle$ is an eigenstate:

$$M_a |u_s\rangle = e^{2\pi i s / r} |u_s\rangle.$$

The eigenvalues are therefore the $r$-th roots of unity, and the phase we want to estimate is

$$\varphi_s = \frac{s}{r}.$$

There is one more crucial identity. Since $a^0 \bmod N = 1$, summing the eigenstates gives

$$\frac{1}{\sqrt r} \sum_{s=0}^{r-1} |u_s\rangle = |1\rangle.$$

So $|1\rangle$ is an **equal superposition of all the eigenstates**. We can prepare it without knowing $r$! Phase estimation on $|1\rangle$ then samples the phases $s/r$, and post-processing recovers $r$.

### Task 3.1: Implement the Eigenstates

Implement `eigenstate(a, N, s, r) -> np.ndarray`.

- `a`, `N`: the base and modulus.
- `s`: the eigenstate index, with $0 \le s < r$.
- `r`: the order of $a$ modulo $N$.
- Returns the length-$2^m$ statevector of $|u_s\rangle$.

> **Hint:** start with a zero vector of length $2^m$. For each $k$ from $0$ to $r-1$, add the amplitude $e^{-2\pi i s k / r} / \sqrt r$ to the entry whose index is $a^k \bmod N$.

In [9]:
def eigenstate(a: int, N: int, s: int, r: int) -> np.ndarray:
    """
    Returns the statevector of the eigenstate |u_s> of M_a.

    Args:
        a: The base, with 1 <= a < N and gcd(a, N) = 1.
        N: The modulus, with N >= 3.
        s: The eigenstate index, with 0 <= s < r.
        r: The order of a modulo N.

    Returns:
        np.ndarray: The length-2**m statevector of |u_s>.
    """
    validate_base(a, N)
    assert isinstance(r, int) and r >= 1, "r must be a positive integer."
    assert isinstance(s, int) and 0 <= s < r, "s must satisfy 0 <= s < r."
    assert pow(a, r, N) == 1, f"r={r} must satisfy a**r = 1 (mod N)."

    # TODO: build the eigenstate from the powers of a modulo N
    m = target_qubit_count(N)
    state = np.zeros(2 ** m, dtype=complex)
    for k in range(r):
        state[pow(a,k,N)] += np.exp(-2j * np.pi * s * k / r) / np.sqrt(r)
    return state

### Task 3.2: Test the Eigenstates

The cell below checks four properties from the lecture:

- each $|u_s\rangle$ is normalized,
- each $|u_s\rangle$ is an eigenstate of $M_a$ with eigenvalue $e^{2\pi i s/r}$,
- the eigenstates are orthonormal, and
- their equal superposition is $|1\rangle$.

In [10]:
EIGENSTATE_CASES = [(2, 5), (3, 5), (2, 15), (2, 21)]

for a, N in EIGENSTATE_CASES:
    r = classical_order(a, N)
    m = target_qubit_count(N)
    matrix = modular_multiplication_matrix(a, N)

    states = []
    for s in range(r):
        state = eigenstate(a, N, s, r)
        assert state.shape == (2 ** m,), (
            f"Eigenstate for a={a}, N={N}, s={s} has shape {state.shape}."
        )
        assert np.isclose(np.linalg.norm(state), 1.0), (
            f"Eigenstate for a={a}, N={N}, s={s} is not normalized."
        )
        assert np.allclose(matrix @ state, np.exp(2j * np.pi * s / r) * state), (
            f"|u_{s}> is not an eigenstate of M_{a} mod {N}."
        )
        states.append(state)

    gram = np.array([[np.vdot(left, right) for right in states] for left in states])
    assert np.allclose(gram, np.eye(r)), (
        f"The eigenstates for a={a}, N={N} are not orthonormal."
    )

    one = np.zeros(2 ** m, dtype=complex)
    one[1] = 1.0
    superposition = sum(states, start=np.zeros(2 ** m, dtype=complex)) / np.sqrt(r)
    assert np.allclose(superposition, one), (
        f"The equal superposition of eigenstates for a={a}, N={N} is not |1>."
    )

print("Eigenstates of M_a passed all tests!")

Eigenstates of M_a passed all tests!


## Task 4: The Order-Finding Circuit

We now use **quantum phase estimation** with the unitary $M_a$. The circuit has two registers:

- a **control register** with $n$ qubits, initially $|0\rangle^{\otimes n}$,
- a **target register** with $m = \lceil \log_2 N \rceil$ qubits, initially $|1\rangle$.

The circuit, matching the lecture's order-finding circuit, is:

1. Prepare the target in $|1\rangle$.
2. Apply a Hadamard to every control qubit.
3. For each control qubit $k$, apply a controlled-$M_a^{2^k}$ to the target register.
4. Apply the inverse QFT to the control register.
5. Measure the control register.

If the target were the eigenstate $|u_s\rangle$, the control measurement would be $y$ with

$$\frac{y}{2^n} \approx \frac{s}{r}.$$

Equivalently, the lecture writes the eigenvalue as $e^{i\pi y / 2^{n-1}}$, which is the same number as $e^{2\pi i y / 2^n}$.

Because the target $|1\rangle$ is a uniform superposition of the $|u_s\rangle$, a single run samples one of the phases $s/r$ uniformly at random. Repeating the circuit and post-processing the outcomes recovers $r$.

### Task 4.1: Implement the Order-Finding Circuit

Implement `order_finding_circuit(a, N, n_count) -> QuantumCircuit`.

- `a`, `N`: the base and modulus.
- `n_count`: the number of control qubits.
- Returns a circuit with `n_count + m` qubits and **no measurements**.

> **Hint:** the controlled power $M_a^{2^k}$ is the modular multiplication matrix for the base $a^{2^k} \bmod N$. Wrap it in a `UnitaryGate` and use `.control(1)`. Append it to `[k] + list(range(n_count, n_count + m))`. Finish by composing `inverse_qft(n_count)` onto the first `n_count` qubits.

In [11]:
def order_finding_circuit(a: int, N: int, n_count: int) -> QuantumCircuit:
    """
    Builds the order-finding circuit for a modulo N.

    Args:
        a: The base, with 1 <= a < N and gcd(a, N) = 1.
        N: The modulus, with N >= 3.
        n_count: The number of control qubits.

    Returns:
        QuantumCircuit: An (n_count + m)-qubit circuit implementing order
        finding, with no measurements.
    """
    validate_base(a, N)
    validate_qubit_count(n_count)

    # TODO: build the order-finding circuit
    m = target_qubit_count(N)
    targets = list(range(n_count, n_count + m))
    circuit = QuantumCircuit(n_count + m)
    circuit.x(n_count)
    circuit.h(range(n_count))
    for k in range(n_count):
        power = pow(a, 2 ** k, N)
        controlled = UnitaryGate(modular_multiplication_matrix(power, N)).control(1)
        circuit.append(controlled, [k] + targets)
    circuit.compose(inverse_qft(n_count), qubits=range(n_count), inplace=True)
    return circuit

### Task 4.2: Test the Order-Finding Circuit

When $r$ divides $2^n$, phase estimation is exact. The control register then contains one of

$$y = \frac{s\,2^n}{r}, \qquad s = 0, 1, \dots, r-1,$$

each with probability $1/r$. The cell below checks this for $r = 4$ and $r = 2$. The target register remains entangled with the control outcome, so we only read out the control register.

In [12]:
CIRCUIT_CASES = [(2, 5, 4), (3, 5, 4), (2, 15, 4), (4, 15, 2)]

for a, N, n_count in CIRCUIT_CASES:
    m = target_qubit_count(N)
    r = classical_order(a, N)
    circuit = order_finding_circuit(a, N, n_count)

    assert circuit.num_qubits == n_count + m, (
        f"Circuit for a={a}, N={N} has {circuit.num_qubits} qubits, "
        f"expected {n_count + m}."
    )
    assert circuit.num_clbits == 0, "The circuit should not contain measurements."

    probabilities = control_probabilities(circuit, n_count)
    expected = {s * 2 ** n_count // r: 1 / r for s in range(r)}

    assert set(probabilities) == set(expected), (
        f"Control outcomes for a={a}, N={N}, n={n_count} are "
        f"{sorted(probabilities)}, expected {sorted(expected)}."
    )
    for outcome, probability in expected.items():
        assert np.isclose(probabilities[outcome], probability), (
            f"Outcome {outcome} has probability {probabilities[outcome]}, "
            f"expected {probability}."
        )

print("Order-finding circuit passed all tests!")

Order-finding circuit passed all tests!


### Caveats from the Lecture

Three practical issues sit between this circuit and a scalable factoring algorithm:

1. **Register size.** The target register must be able to hold every value $0, \dots, N-1$. Since $N$ is not necessarily a power of two, $M_a$ is defined to act as the identity on the unused basis states.
2. **Controlled powers.** The circuit needs controlled-$M_a^{2^k}$ for $k = 0, \dots, n-1$. Applying $M_a$ one power at a time would require exponentially many applications, but repeated squaring computes each $a^{2^k} \bmod N$ in polynomially many steps. Here we build each controlled power directly from a matrix; a scalable implementation instead compiles modular multiplication into elementary gates, which is where most of the engineering effort in real Shor implementations goes.
3. **Number of control qubits.** More control qubits give a more accurate estimate of $s/r$. For the small examples in this lab, `target_qubit_count(N) + 3` control qubits is enough. In general one uses about $2 \lceil \log_2 N \rceil$ control qubits to guarantee a good approximation with high probability.

## Task 5: Post-Processing with Continued Fractions

The control register does not usually produce $r$ directly. It produces an integer $y$ such that

$$\frac{y}{Q} \approx \frac{s}{r}, \qquad Q = 2^{n_{\text{count}}}.$$

We need to recover $r$ from this **approximation**. The tool is the **continued-fraction expansion**.

A continued fraction is written

$$[a_0, a_1, \dots, a_M] = a_0 + \cfrac{1}{a_1 + \cfrac{1}{a_2 + \cfrac{1}{\ddots + \cfrac{1}{a_M}}}}.$$

The **$m$-th convergent** is the rational number $[a_0, \dots, a_m]$. The lecture quotes the standard theorem:

> If $\left|\frac{s}{r} - \varphi\right| \le \frac{1}{2r^2}$, then $\frac{s}{r}$ is a convergent of the continued fraction for $\varphi$.

With enough control qubits, $\varphi = y/Q$ satisfies this bound, so one of the convergents has denominator $r$ (or a divisor of $r$, if $s$ and $r$ share a factor).

### Example from the Lecture

Suppose a function has order $5$, and the measurement gives $y = 307$ out of $Q = 512$. (The lecture uses $10$ control qubits and divides by $2^9 = 512$; in Qiskit's convention this is a $9$-bit estimate with $Q = 2^9$.)

$$\frac{307}{512} = [0, 1, 1, 2, 102]
\quad\Longrightarrow\quad
\text{convergents } 0,\ 1,\ \frac{1}{2},\ \frac{3}{5},\ \frac{307}{512}.$$

The denominator $5$ is the order.

### Task 5.1: Implement the Convergents

Implement `continued_fraction_convergents(numerator, denominator) -> list[Fraction]`.

- Returns every convergent of `numerator / denominator`, in order, including the zeroth convergent.
- Use `fractions.Fraction` for exact rational arithmetic.

> **Hint:** run the Euclidean algorithm. If the current quotient is $a$, update the numerator and denominator of the convergent with
> $$p_m = a_m p_{m-1} + p_{m-2}, \qquad q_m = a_m q_{m-1} + q_{m-2},$$
> starting from $p_{-2}=0$, $p_{-1}=1$, $q_{-2}=1$, $q_{-1}=0$.

In [ ]:
def continued_fraction_convergents(numerator: int, denominator: int) -> list[Fraction]:
    """
    Returns the convergents of numerator / denominator.

    Args:
        numerator: A nonnegative integer.
        denominator: A positive integer.

    Returns:
        list[Fraction]: The convergents, from the zeroth onward.
    """
    assert isinstance(numerator, int) and numerator >= 0, (
        "numerator must be a nonnegative integer."
    )
    assert isinstance(denominator, int) and denominator > 0, (
        "denominator must be a positive integer."
    )

    # TODO: run the Euclidean algorithm and build the convergents
    pass

### Task 5.2: Recover the Order from One Measurement

Implement `order_from_measurement(a, N, y, n_count) -> int | None`.

- `y` is a measured control-register outcome.
- `n_count` determines $Q = 2^{n_{\text{count}}}$.
- Examine the convergents of $y/Q$. Return the smallest denominator $r > 1$ that satisfies `pow(a, r, N) == 1`.
- If no convergent gives a valid order, return `None`.

> **Hint:** use `reduce_order` on each candidate denominator before testing it. Some convergents may have a denominator that is a multiple of the true order, or a denominator that does not satisfy $a^r \equiv 1$.

In [ ]:
def order_from_measurement(a: int, N: int, y: int, n_count: int) -> int | None:
    """
    Recovers the order from one control-register measurement.

    Args:
        a: The base, with 1 <= a < N and gcd(a, N) = 1.
        N: The modulus, with N >= 3.
        y: The measured control-register outcome.
        n_count: The number of control qubits.

    Returns:
        int | None: The recovered order, or None if no convergent works.
    """
    validate_base(a, N)
    validate_qubit_count(n_count)
    assert isinstance(y, int) and 0 <= y < 2 ** n_count, (
        f"y must be an integer in [0, 2**n_count), got {y}."
    )

    # TODO: use the convergents of y / 2**n_count to find a valid order
    pass

### Task 5.3: Test the Continued-Fraction Post-Processing

The cell below checks the lecture's $307/512$ example, the full convergent list, and order recovery for several bases and moduli.

In [ ]:
expected_convergents = [
    Fraction(0, 1),
    Fraction(1, 1),
    Fraction(1, 2),
    Fraction(3, 5),
    Fraction(307, 512),
]
observed_convergents = continued_fraction_convergents(307, 512)
assert observed_convergents == expected_convergents, (
    f"Convergents of 307/512 are {observed_convergents}, "
    f"expected {expected_convergents}."
)

assert order_from_measurement(2, 31, 307, 9) == 5, (
    "The lecture example with y=307 and Q=512 should recover order 5."
)

for a, N in [(2, 5), (2, 7), (2, 9), (2, 15), (4, 15), (2, 21), (5, 21)]:
    r = classical_order(a, N)
    n_count = target_qubit_count(N) + 3
    Q = 2 ** n_count

    # Pick the exact outcome for s = 1. When r does not divide Q, use the
    # nearest representable phase and let the continued fraction do its job.
    y = round(Q / r)
    observed = order_from_measurement(a, N, y, n_count)
    assert observed == r, (
        f"order_from_measurement(a={a}, N={N}, y={y}, n={n_count}) "
        f"returned {observed}, expected {r}."
    )

assert order_from_measurement(2, 15, 1, 4) is None, (
    "A measurement that does not approximate a valid phase should return None."
)

print("Continued-fraction post-processing passed all tests!")

## Task 6: Quantum Order Finding

We now combine the pieces. One run of the order-finding circuit produces a control outcome $y$. Post-processing that outcome gives a candidate order. Because the target $|1\rangle$ is a superposition of all $|u_s\rangle$, different runs can produce different values of $s$, and some of those values share factors with $r$.

There are two standard ways to handle this:

- Run the circuit several times and take the **least common multiple** of the recovered denominators. If two runs give $r/\gcd(s_1,r)$ and $r/\gcd(s_2,r)$ with coprime cofactors, their lcm is $r$.
- In exact simulation, examine **all** control outcomes with nonzero probability. Every outcome that survives the continued-fraction test is a multiple or divisor of $r$; reducing each candidate and taking the smallest valid order recovers $r$.

The cell below takes the second approach, which is deterministic and avoids shot noise.

### When could it fail?

- **Bad estimate of $s/r$.** This becomes negligible with more control qubits.
- **A common factor between $s$ and $r$.** The continued fraction then returns $r / \gcd(s,r)$, a divisor of the true order. Repeating the experiment, or reducing candidates as above, fixes it.

### Task 6.1: Implement Quantum Order Finding

Implement `quantum_order(a, N, n_count=None) -> int`.

- Use `order_finding_circuit` and `control_probabilities`.
- For every outcome with nonzero probability, call `order_from_measurement`.
- Collect the valid candidates and return the smallest one.
- If `n_count` is `None`, use `target_qubit_count(N) + 3`.
- Handle the special case $a \equiv 1 \pmod N$, whose order is $1$.

> **Hint:** outcomes with probability below `1e-12` are numerical noise; skip them. The true order is the smallest integer $r$ with `pow(a, r, N) == 1`, so taking the minimum over all valid candidates is safe.

In [ ]:
def quantum_order(a: int, N: int, n_count: int | None = None) -> int:
    """
    Finds the order of a modulo N with quantum phase estimation.

    Args:
        a: The base, with 1 <= a < N and gcd(a, N) = 1.
        N: The modulus, with N >= 3.
        n_count: The number of control qubits. Defaults to
            target_qubit_count(N) + 3.

    Returns:
        int: The order of a modulo N.
    """
    validate_base(a, N)

    # TODO: run the order-finding circuit, post-process every outcome with
    # continued fractions, and return the smallest valid order
    pass

### Task 6.2: Test Quantum Order Finding

The cell below compares `quantum_order` with the classical routine for a range of bases and moduli. The moduli stay small enough that exact statevector simulation is fast.

In [ ]:
ORDER_CASES = [
    (3, 5),
    (2, 7),
    (2, 9),
    (4, 9),
    (2, 15),
    (4, 15),
    (2, 21),
    (4, 21),
    (5, 21),
]

for a, N in ORDER_CASES:
    expected = classical_order(a, N)
    observed = quantum_order(a, N)
    assert observed == expected, (
        f"quantum_order(a={a}, N={N}) returned {observed}, expected {expected}."
    )

print("Quantum order finding passed all tests!")

## Task 7: Factoring from the Order

The final classical step turns an order into a factor. Suppose $r$ is the order of $a$ modulo $N$ and $r$ is even. Set

$$x = a^{r/2} \bmod N.$$

Then

$$x^2 \equiv a^r \equiv 1 \pmod N,$$

so $N$ divides

$$x^2 - 1 = (x - 1)(x + 1).$$

Unless $x \equiv \pm 1 \pmod N$, the two factors $x-1$ and $x+1$ split the prime factors of $N$ between them, and

$$d = \gcd(x - 1, N)$$

is a nontrivial factor of $N$.

For example, with $a = 2$ and $N = 15$, the order is $r = 4$, so $x = 2^2 = 4$ and

$$\gcd(4 - 1, 15) = \gcd(3, 15) = 3.$$

The other factor is $15 / 3 = 5$.

### Task 7.1: Implement Factoring from the Order

Implement `factor_from_order(N, a, r) -> int | None`.

- If $r$ is odd, return `None`.
- Compute $x = a^{r/2} \bmod N$.
- Compute $d = \gcd(x - 1, N)$.
- Return $d$ if $1 < d < N$; otherwise return `None`.

> **Hint:** use `pow(a, r // 2, N)` so that the modular exponentiation stays exact and fast.

In [ ]:
def factor_from_order(N: int, a: int, r: int) -> int | None:
    """
    Returns a nontrivial factor of N given the order r of a modulo N.

    Args:
        N: The modulus, with N >= 3.
        a: The base, with 1 <= a < N and gcd(a, N) = 1.
        r: The order of a modulo N.

    Returns:
        int | None: A nontrivial factor of N, or None if this order fails.
    """
    validate_base(a, N)
    assert isinstance(r, int) and r >= 1, "r must be a positive integer."

    # TODO: apply the order-to-factor reduction
    pass

### Task 7.2: Test Factoring from the Order

The cell below checks the lecture's example and several other bases, including orders that fail the reduction.

In [ ]:
FACTOR_CASES = [
    (15, 2, 4, 3),
    (15, 4, 2, 3),
    (21, 2, 6, 7),
    (35, 2, 12, 7),
    (55, 2, 20, 11),
]

for N, a, r, expected in FACTOR_CASES:
    observed = factor_from_order(N, a, r)
    assert observed is not None, (
        f"factor_from_order(N={N}, a={a}, r={r}) returned None."
    )
    assert observed in {expected, N // expected}, (
        f"factor_from_order(N={N}, a={a}, r={r}) returned {observed}, "
        f"expected {expected} or {N // expected}."
    )

assert factor_from_order(21, 4, 3) is None, (
    "An odd order should not produce a factor."
)
assert factor_from_order(15, 2, 2) is None, (
    "An invalid order should not produce a factor."
)

print("Factoring from the order passed all tests!")

## Task 8: The Full Algorithm

Factoring a general $N$ adds a few classical checks before order finding is needed. From the lecture:

- **Given number $N$**, find two factors, then recursively factor each of them.
- **First tactics:**
  1. Test whether $N$ is prime (for example, with a Miller-Rabin test).
  2. If $N$ is even, then $2$ is a factor.
  3. Check whether $N$ is a perfect power $b^e$. This takes at most $\log N$ steps.

If none of these applies, $N$ is odd, composite, and not a prime power. Choose a base $a \in \{2, \dots, N-2\}$, compute $d = \gcd(a, N)$ (if $d > 1$, we are already done), and use quantum order finding on $a$. If the resulting order does not split $N$, retry with a different base.

Each attempt succeeds with probability at least $1/2$, so $t$ attempts succeed with probability at least $1 - 2^{-t}$.

### Task 8.1: Implement the Full Factoring Algorithm

Implement `shor(N) -> list[int]`.

- Returns the prime factors of $N$ in nondecreasing order, **with multiplicity**, so `shor(8)` returns `[2, 2, 2]`.
- Use `is_prime` and `perfect_power` for the classical preprocessing.
- Use `quantum_order` and `factor_from_order` for the order-finding step.
- If a base fails to split $N$, try the next base.

> **Hint:** maintain a stack of integers still to be factored. When an integer is prime, append it to the answer. When it is even, append `2` and push `n // 2`. When it is a perfect power, push its base and the cofactor. Otherwise, try bases `2, 3, 4, ...`. If `gcd(a, n) > 1`, that gcd is already a factor. Otherwise use the order of `a` and `factor_from_order`, and push both pieces once a nontrivial factor is found.

In [ ]:
def shor(N: int) -> list[int]:
    """
    Factors N into primes using Shor's algorithm.

    Args:
        N: The integer to factor, with N >= 3.

    Returns:
        list[int]: The prime factors of N in nondecreasing order, with
        multiplicity.
    """
    validate_modulus(N)

    # TODO: recursively split N into prime factors
    pass

### Task 8.2: Final Test

The cell below factors a mix of even numbers, prime powers, primes, and odd composite numbers. It must pass for the lab to be complete.

> **Note:** the final two composite cases are larger and take noticeably longer to simulate.

In [ ]:
FACTORING_CASES = [
    (4, [2, 2]),
    (8, [2, 2, 2]),
    (9, [3, 3]),
    (13, [13]),
    (15, [3, 5]),
    (21, [3, 7]),
    (25, [5, 5]),
    (27, [3, 3, 3]),
    (35, [5, 7]),
    (49, [7, 7]),
    (55, [5, 11]),
]

for N, expected in FACTORING_CASES:
    observed = shor(N)
    assert observed == expected, (
        f"shor({N}) returned {observed}, expected {expected}."
    )

print("Shor's algorithm passed all tests!")